# Ngày 4 · Gradient descent

- Tính gradient MSE và kiểm tra bằng sai phân
- Giải thích tác động của learning rate

**Đọc:**
- Stanford CS229, §1.1–1.2
- UC Berkeley CS189, Gradient Descent

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge

DATA = next((p for p in (Path('data'), Path('../data')) if (p/'real_estate.csv').is_file()), None)
if DATA is None:
    raise FileNotFoundError('Mở notebook mẫu từ thư mục py30 hoặc bai_lam.')
real_estate = pd.read_csv(DATA / 'real_estate.csv')

def _check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, 'Chưa cài đặt (NotImplementedError)'
    except Exception as error:
        ok, detail = False, f'{type(error).__name__}: {error}'
    return {'name': name, 'ok': bool(ok), 'detail': str(detail)}

def _report(ex_id, *checks):
    print('__PY30__' + json.dumps({'id': ex_id, 'checks': list(checks)}, ensure_ascii=False))
x = np.log1p(real_estate.mrt_distance_m.to_numpy())
_X = np.column_stack([np.ones(len(x)),(x-x.mean())/x.std()])
_y=real_estate.price_10k_ntd_per_ping.to_numpy()
_w=np.zeros(2)
_expected=2*_X.T@(_X@_w-_y)/len(_y)
_opt=np.linalg.lstsq(_X,_y,rcond=None)[0]

def _keeps_w(fn):
    w0 = np.zeros(2); fn(_X, _y, w0, .1, 5)
    return np.array_equal(w0, np.zeros(2))


## Gradient và kiểm tra số

Gợi ý: Đánh giá loss ở hai phía của từng tọa độ, dùng bản sao.

In [ ]:
def grad_mse(X,y,w):
    raise NotImplementedError

def numerical_grad(loss,w,eps=1e-6):
    raise NotImplementedError

def gradient_descent(X,y,w,lr,steps):
    raise NotImplementedError


In [ ]:
_report('d4-gradient',
    _check('Gradient giải tích và gradient số đúng', lambda: (np.allclose(grad_mse(_X,_y,_w),_expected) and np.allclose(numerical_grad(lambda w:np.mean((_X@w-_y)**2),_w),_expected,atol=1e-6), 'Đối chiếu cả hai với gradient tham chiếu.')),
    _check('Hội tụ về nghiệm đóng', lambda: (np.allclose(gradient_descent(_X,_y,_w,.1,300),_opt,atol=1e-6), 'Chạy từ cùng điểm đầu.')),
    _check('Không sửa w ban đầu', lambda: (_keeps_w(gradient_descent), 'Làm việc trên bản sao của w.')),
)


## Dự án

So nghiệm GD và nghiệm đóng trên UCI sau khi chuẩn hóa đặc trưng trên train. Ghi trường hợp loss dao động khi tăng learning rate.